# Warehouse basics — read, check, drop

Three small things you do all the time, once a feed is loaded:

1. **Read a table with Polars** — partitioned and not, whole and filtered
2. **Check the files** behind a table — where they are, how big, all still there
3. **Drop a table** — and the difference between dropping it and deleting its data

Self-contained: it builds its own drop in `_notebook_wh/` and the last cell
deletes it. Nothing of yours is touched.

In [ ]:
import shutil, sys, zipfile
from pathlib import Path

import polars as pl

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
pl.Config(fmt_str_lengths=44, tbl_rows=12)

WORK = ROOT / "_notebook_wh"
shutil.rmtree(WORK, ignore_errors=True)
(WORK / "drops").mkdir(parents=True)
WAREHOUSE = WORK / "ws" / "warehouse"

print("project   :", ROOT)
print("scratch   :", WORK)

---
## 0. One drop, two tables

Four days in one zip, one member per day — which is the shape the runbook assumes:
**one archive, one business date per file.** We load it twice, into two tables that
differ in exactly one setting: `target.partition_by`.

In [ ]:
from ffe.core.spec import FeedSpec
from ffe.io.runner import run

DAYS = ["20260817", "20260818", "20260819", "20260820"]
MEMBER = '''F|issue.{day}|{day}
H|{day}|issue|id|name
I|{a}|Abhishek
I|{b}|Nilanjana
I|{c}|Ankita
E|issue
'''

drop = WORK / "drops" / "issues_week.zip"
with zipfile.ZipFile(drop, "w", zipfile.ZIP_DEFLATED) as z:
    for i, day in enumerate(DAYS):
        z.writestr(f"issue_{day}.txt", MEMBER.format(day=day, a=i * 3 + 1, b=i * 3 + 2, c=i * 3 + 3))


def load(table: str, partition_by: list[str]):
    spec = FeedSpec.from_yaml(ROOT / "feeds" / "pipe-tagged-feed.yaml")
    spec.source.pattern = str(drop)
    spec.target.table = table
    spec.target.partition_by = partition_by
    job = run(spec, WORK / "ws", workers=4)
    print(f"{table:22} {job.status:6} rows={job.rows:3}  files={job.commit['files']}  "
          f"partition_by={partition_by or '(none)'}")
    return job

load("demo.by_day", ["business_date"])
load("demo.flat", [])

---
## 1. Reading it back with Polars

Two ways in. `scan()` is the convenience wrapper in `ffe.io.sink` — whole table,
straight into a DataFrame. Fine for a table this size; **it reads everything.**

In [ ]:
from ffe.io.sink import scan

df = scan(WAREHOUSE, "demo.by_day")
print(f"{len(df)} rows, {len(df.columns)} columns")
print(df.select(["id", "name", "business_date", "_src_file"]))

In [ ]:
# rows per day -- the same answer from either table
for name in ("demo.by_day", "demo.flat"):
    counts = scan(WAREHOUSE, name).group_by("business_date").len().sort("business_date")
    print(name, dict(zip(counts["business_date"], counts["len"])))

The other way is to push the filter **down into Iceberg** so it only opens the files
it needs. `plan_files()` says how many that is, before any data is read — which is
the cheap way to see whether your filter is actually doing anything.

In [ ]:
from ffe.io.sink import catalog

cat = catalog(WAREHOUSE)
DAY = "20260819"

for name in ("demo.by_day", "demo.flat"):
    table = cat.load_table(name)
    all_files = len(list(table.scan().plan_files()))
    one_day = len(list(table.scan(row_filter=f"business_date == '{DAY}'").plan_files()))
    rows = pl.from_arrow(table.scan(row_filter=f"business_date == '{DAY}'").to_arrow())
    by_id = {f.field_id: f.name for f in table.schema().fields}
    grain = [by_id.get(f.source_id) for f in table.spec().fields] or "(unpartitioned)"
    print(f"{name:14} {str(grain):20} files: {one_day}/{all_files} opened   "
          f"rows for {DAY}: {len(rows)}")

**Both tables open 1 file out of 4.** That is worth understanding rather than
glossing over:

- `add_files` reads each Parquet footer at commit time, so Iceberg already knows the
  min/max `business_date` **per file** — even with no partition spec at all. Here
  every member holds exactly one day, so the stats alone are enough to skip 3 files.
- What `partition_by` buys you is that this stops being luck. The value is recorded
  in the manifest as the file's partition, so pruning holds however the data is
  laid out, and the table advertises its grain to anything reading it.
- It also turns a mistake into a refusal: `add_files` will not register a file that
  spans two partition values, so a zip that quietly mixes two business dates fails
  the load instead of landing skewed. That's the `mixed_partition_file` error.

So: partition when the grain is real and you want it enforced — not to make small
filtered reads faster, which footer stats already do.

In [ ]:
# what the partition actually looks like, file by file
files = cat.load_table("demo.by_day").inspect.files()
print(pl.from_arrow(files.select(["partition", "record_count", "file_size_in_bytes"])))
print("\nand for the unpartitioned table:")
print(pl.from_arrow(cat.load_table("demo.flat").inspect.files().select(["partition", "record_count"])))

---
## 2. Checking the files behind a table

Where does the data actually live? Not where you'd guess.

In [ ]:
print("under the warehouse directory:")
for path in sorted(WAREHOUSE.rglob("*")):
    if path.is_file():
        print("   ", path.relative_to(WAREHOUSE))

print("\nwhere demo.by_day's data files are:")
for rec in cat.load_table("demo.by_day").inspect.files().to_pylist():
    print("   ", rec["file_path"].replace(WORK.as_uri() + "/", ""), f"({rec['record_count']} rows)")

The warehouse holds **metadata only** — `catalog.db`, some JSON, some Avro manifests.
The Parquet is still sitting in `staging/<job_id>/good/`, exactly where the workers
wrote it, because `add_files` registers files *in place* rather than copying rows.

That's what makes the commit cheap, and it's also the one thing to know before
tidying up: **staging is not scratch space once a job has committed.** Delete it and
the table still lists its files, but reading it fails.

A basic health check, then, is: does every file the table claims still exist?

In [ ]:
def check(table_name: str) -> dict:
    table = cat.load_table(table_name)
    rows = table.inspect.files().to_pylist()
    paths = [Path(r["file_path"].replace("file://", "")) for r in rows]
    missing = [p for p in paths if not p.exists()]
    snap = table.current_snapshot()
    return {
        "table": table_name,
        "files": len(paths),
        "missing": len(missing),
        "rows": int(snap.summary.get("total-records", 0)) if snap else 0,
        "bytes_claimed": sum(r["file_size_in_bytes"] for r in rows),
        "bytes_on_disk": sum(p.stat().st_size for p in paths if p.exists()),
        "first_missing": str(missing[0]) if missing else None,
    }

for name in ("demo.by_day", "demo.flat"):
    print(check(name))

In [ ]:
# break it on purpose: someone "cleaned up" staging
victim = Path(cat.load_table("demo.flat").inspect.files().to_pylist()[0]["file_path"].replace("file://", ""))
victim.unlink()

print("check says:", check("demo.flat"))
try:
    scan(WAREHOUSE, "demo.flat")
except FileNotFoundError as exc:
    print("\nreading it now:", type(exc).__name__, str(exc)[:90], "...")

The check catches it as a count; the read catches it as a `FileNotFoundError` halfway
through. Prefer the first. Run it after anything that moves or prunes files, and
before trusting a table you haven't read in a while.

---
## 3. Dropping a table

`demo.flat` is broken now, so it's a fair thing to throw away. Two verbs, and the
difference matters:

| | catalog entry | Parquet on disk |
|---|---|---|
| `drop_table` | gone | **left behind** |
| `purge_table` | gone | deleted |

In [ ]:
print("exists before:", cat.table_exists("demo.flat"))

remaining = [p for p in [Path(r["file_path"].replace("file://", ""))
                         for r in cat.load_table("demo.flat").inspect.files().to_pylist()]
             if p.exists()]

cat.drop_table("demo.flat")                    # metadata only

print("exists after :", cat.table_exists("demo.flat"))
print("its parquet still on disk:", sum(p.exists() for p in remaining), "of", len(remaining))

So `drop_table` on its own leaves orphans. That is often what you want — re-registering
the same files into a fixed table is a second of work, whereas re-parsing the drop is not.

`purge_table` is the other choice: it drops **and** deletes the data files. Here those
files live in staging, so this really does reach into `staging/<job_id>/good/` and
remove Parquet.

In [ ]:
staged_before = sorted((WORK / "ws" / "staging").rglob("*.parquet"))

if cat.table_exists("demo.by_day"):            # the "if needed" part
    cat.purge_table("demo.by_day")

staged_after = sorted((WORK / "ws" / "staging").rglob("*.parquet"))
print(f"staged parquet: {len(staged_before)} -> {len(staged_after)}")
print("tables left:", [f"{ns}.{n}" for (ns,) in cat.list_namespaces() for _, n in cat.list_tables(ns)])

One caution that follows from files-registered-in-place: if you ever load the *same*
staged files into two tables, purging one deletes the data out from under the other.
`ffe run` gives every job its own staging directory, so this only bites if you
re-register files by hand.

Re-running a feed, by the way, does **not** replace anything — it appends a second
copy. That's [`docs/RUNBOOK.md`](../docs/RUNBOOK.md), and dropping the table first is
one of the ways out.

---
## 4. Clean up

Deletes only `_notebook_wh/` — which here means the drops, the staging, the ledger and
the warehouse, since everything this notebook made lives under it.

In [ ]:
shutil.rmtree(WORK, ignore_errors=True)
print("cleaned up", WORK)

---
### See also

- **[`notebooks/explore.ipynb`](explore.ipynb)** — the whole path, messy file to table
- **[`docs/RUNBOOK.md`](../docs/RUNBOOK.md)** — the real drop, the sandbox trial, double-loading
- **`src/ffe/io/sink.py`** — `commit`, `scan`, and why partitioning is refused on a populated table